In [ ]:
from megatron.core.parallel_state import create_group
from .utils import GlobalMemoryBuffer, is_torch_min_version


def create_group(
    ranks=None,
    timeout=None,
    backend=None,
    pg_options=None,
    use_local_synchronization=False,
    group_desc=None,
):
    """Creates a ProcessGroup."""
    kwargs = {
        "ranks": ranks,
        "timeout": timeout,
        "backend": backend,
        "pg_options": pg_options,
        "use_local_synchronization": use_local_synchronization,
        "group_desc": group_desc,
    }
    '''
    低版本兼容设置：
    '''
    if not is_torch_min_version("2.4.0"):
        kwargs.pop("group_desc")
        if timeout is None:
            # Old version (e.g. v2.1.2) sets default_pg_timeout as default value to timeout
            # in function signature, then check tiemout value type.
            # New version sets None as default value to timeout in function signature. If value
            # is None, torch will give value according to the backend, then check type.
            # So need to unset timeout here if caller doesn't set value. Otherwise there is
            # type error.
            kwargs.pop("timeout")
            
    group = torch.distributed.new_group(**kwargs)
    global _global_process_group_list
    if _global_process_group_list is None:
        # None stands for the default process group
        _global_process_group_list = [None]
        
    if torch.distributed.get_rank() in ranks:
        _global_process_group_list.append(group)
    return group


# create_group
- 这段代码是一个基于 PyTorch 分布式通信库（torch.distributed）的封装函数，<font color='red'>主要用于在分布式训练环境中创建和管理自定义的进程组（ProcessGroup）。</font>

- 在分布式训练中，<font color='red'>默认的进程组（default_pg）包含所有参与训练的进程（即全局的 world）。</font>
  - <font color='green'>而 create_group 的作用是从全局进程中挑选出一部分进程（ranks）</font>，组建一个独立的通信小组，以便进行更细粒度的并行策略（如模型并行、流水线并行）或特定的集合通信操作。

下面为你详细拆解这段代码的逻辑和各个参数的含义：

## 🛠️ 核心逻辑与参数解析
#### 1. 参数准备与版本兼容处理：

函数首先将所有入参打包进 kwargs 字典，然后进行了一次重要的版本兼容性检查：
 - group_desc 参数：<font color='red'>如果当前 PyTorch 版本低于 2.4.0，会移除 group_desc（进程组描述）参数，因为旧版本不支持。</font>
 - timeout 参数：这是一个非常细节的兼容性处理。旧版本的 PyTorch 在函数签名中直接使用了具体的超时时间作为默认值，而新版本（2.4.0及以上）默认是 None（内部会根据后端自动分配）。为了防止在旧版本中传入 None 导致类型错误，代码在旧版本且用户未指定 timeout 时，主动将其从参数中移除。
#### 2. 调用底层 API：
 - group = <font color='red'>torch.distributed.new_group(**kwargs)</font>
   -  <font color='blue'>这是真正执行创建进程组的 PyTorch 原生接口。</font>

#### 3. 全局进程组列表管理：
- 代码维护了一个全局列表 _global_process_group_list，<font color='red'>用于记录当前进程中所有创建过的进程组。</font>
- if torch.distributed.get_rank() in ranks:：<font color='red'>这是一个关键的保护机制。在分布式环境中，创建进程组时，只有属于该组（即 rank 在 ranks 列表中）的进程才会实际参与创建并保存该 group 对象。不属于该组的进程不会将其加入自己的全局列表中。</font>

## 📝 参数详细说明
- ranks (list): 组成该进程组的进程编号（rank）列表。例如 [0, 1, 2, 3] 表示由 0 到 3 号进程组成一个小组。
- timeout (timedelta): 该进程组内集合通信操作的超时时间。
- backend (str): 通信后端，如 'nccl' (常用于 NVIDIA GPU)、'gloo' (常用于 CPU) 等。如果不指定，通常沿用默认进程组的后端。
- pg_options: <font color='red'>针对特定后端的初始化选项（例如 NCCL 的分块大小等高级配置）。</font>
- use_local_synchronization (bool): 是否使用本地同步。通常用于优化某些特定硬件或拓扑结构下的同步效率。
  - 含义：是否使用本地同步机制。
  - 解释：默认情况下，创建一个新组时，所有参与的进程都需要进行全局范围的同步以确保状态一致。
  - <font color='red'>如果将其设置为 True，则只要求属于该组的局部进程进行同步。</font>这在某些复杂的多组拓扑结构或动态创建组时可以显著提高启动速度并减少不必要的通信开销。
  
- group_desc (str): 进程组的描述信息（仅在 PyTorch >= 2.4.0 时生效，方便调试和追踪）。

## 💡 总结
- 这个 create_group 函数不仅是对 torch.distributed.new_group 的简单调用，<font color='red'>还额外增加了版本兼容性适配和全局状态管理</font>。它确保了在不同 PyTorch 版本下都能稳定创建自定义的通信域，并方便开发者在代码的其他地方统一管理和调用这些进程组。

## pg_options

在分布式训练（如 PyTorch 的 torch.distributed）中，pg_options 是一个用于向特定的通信后端（Backend）传递高级、底层配置参数的对象。

结合你上一轮提到的代码上下文，这里的 pg_options 特指 PyTorch 分布式进程组选项。最常见的场景是当后端为 'nccl' 时，它通常是一个 ProcessGroupNCCL.Options 类的实例。

### 一、 pg_options 的核心作用
默认的 NCCL 或 Gloo 后端已经有一套经过优化的默认配置，足以应对大多数训练任务。<font color='red'>但在以下复杂场景中，你需要通过 pg_options 进行微调：</font>

1. 网络拓扑控制：<font color='red'>指定使用哪个网卡接口进行节点间通信。</font>
2. 性能调优：<font color='red'>调整内部缓冲区大小、开启/关闭特定的优化算法（如异步错误处理）。</font>
3. 调试与容错：<font color='red'>设置更严格的超时检查、开启详细的 NCCL 日志等。</font>

### 二、 具体举例说明

#### 示例 1：指定通信使用的网卡接口 (Network Interface)
在多网卡服务器上，如果不加干预，NCCL 可能会选择错误的网口导致通信极慢甚至失败。可以通过 pg_options 强制指定：
- set_network_interface

In [ ]:
import torch.distributed as dist

# 创建 NCCL 选项对象
options = dist.ProcessGroupNCCL.Options()
# 强制指定使用 eth0 和 ib0 进行通信
options.set_network_interface("eth0,ib0") 

dist.init_process_group(
    backend="nccl", 
    pg_options=options
)

1. eth0 (Ethernet Interface 0)
- 含义：eth 是 Ethernet（以太网）的缩写，0 表示系统中的第一块以太网卡。它是传统且最常见的网络接口命名方式。
- 功能与特点：主要用于常规的局域网（LAN）或广域网（WAN）通信。它基于标准的以太网协议，通常通过 RJ45 网线连接到交换机或路由器，负责收发常规的网络数据包（如 SSH 管理流量、HTTP 请求等）。
- 命名演变：需要注意的是，在较新的 Linux 发行版（如 CentOS 7+ 或现代 Ubuntu）中，由于引入了可预测的命名规则，传统的 eth0 往往已被更具描述性的名称取代，例如 enp0s3（PCI-E插槽网卡）、ens33（主板集成网卡）等。
2. ib0 (InfiniBand Interface 0)
- 含义：ib 是 InfiniBand 的缩写，0 同样代表第一块该类型的网卡。
- 功能与特点：InfiniBand 是一种专为高性能计算（HPC）和数据中心设计的网络通信技术。与普通以太网相比，它具有极高的带宽和极低的延迟。此外，它原生支持 RDMA（远程直接内存访问）技术，允许一台计算机直接读写另一台计算机的内存，而无需经过操作系统的内核干预。
- 应用场景：在大型 AI 训练集群、超级计算机中，节点间庞大的 GPU 数据同步（如 PyTorch 中的 NCCL 集合通信）几乎全部依赖 InfiniBand 网络来完成。

##### 💡 结合您之前的上下文总结
在多网卡的服务器上进行分布式训练时，通常会同时存在这两种接口。合理的分工策略是：
- 使用 eth0（或以太网口）：<font color='red'>处理日常运维、代码拉取、日志记录、心跳检测等轻量级业务流量。</font>
- 使用 ib0（或 InfiniBand/RoCE 接口）：<font color='blue'>专门用于多机多卡之间的模型参数同步和数据交换，以最大化 GPU 的计算利用率并避免通信瓶颈。</font>

##### 网络接口
在 Linux 下，推荐使用现代命令或传统命令来查看网络接口：
1. 使用 ip 命令（推荐）
<font color='blue'>这是目前主流的 Linux 发行版中最推荐的工具。</font>
- 查看所有接口的 IP 信息：输入 ip addr show 或简写 ip a。
- 仅查看接口状态和名称：输入 ip link show。
- 彩色或简洁输出：可以使用 ip -c addr show（彩色高亮）或 ip -br addr show（简洁模式）。
2. 使用 ifconfig 命令（传统方式）
- 查看已激活的接口：直接输入 ifconfig。
- 查看所有接口（包含未激活的）：输入 ifconfig -a。
- 注：在某些新系统中，该命令可能已被弃用，若提示找不到命令，请使用上述的 ip 命令替代。

#### 示例 2：调整超时时间 (Timeouts)
虽然 new_group() 有独立的 timeout 参数，但底层的 NCCL 也有自己的超时机制。如果训练中存在长时间的计算（如大模型前向传播），可以延长底层等待时间以防误报：
- _timeout

In [ ]:
import datetime
import torch.distributed as dist

options = dist.ProcessGroupNCCL.Options()
# 将底层操作超时时间设置为 30 分钟
options._timeout = datetime.timedelta(minutes=30)

dist.init_process_group(
    backend="nccl", 
    pg_options=options
)

### 示例 3：启用异步错误处理 (Asynchronous Error Handling)
在某些大规模集群中，如果某个 GPU 发生硬件故障，同步报错可能会导致整个训练卡死。开启异步错误处理可以让 NCCL 在后台检测并抛出异常，而不阻塞其他正常运行的进程：

In [ ]:
options = dist.ProcessGroupNCCL.Options()
# 开启异步错误处理机制
options.is_high_priority_stream = True 
# 注：不同版本的 PyTorch 提供的 API 可能略有差异，部分版本直接支持 options.async_error_handling = True

dist.init_process_group(backend="nccl", pg_options=options)

### 💡 补充提示：注意区分 PostgreSQL 的 PGOPTIONS
- 由于缩写相似，如果你是在做数据库开发而非深度学习，PostgreSQL 也有一个名为 PGOPTIONS 的环境变量。它的作用是在启动客户端会话时传递参数（例如 env PGOPTIONS="-c geqo=off -c statement_timeout=5min" psql），用来修改当前会话的查询优化器行为或超时限制。但这与 PyTorch 分布式训练中的 pg_options 完全是两个不同的概念。

# torch.distributed.new_group(**kwargs)

- torch.distributed.new_group(**kwargs) 是 PyTorch 分布式通信包（torch.distributed）中用于创建自定义进程组（ProcessGroup）的核心接口。
- 在分布式训练中，<font color='red'>系统默认会创建一个包含所有进程的全局进程组（通常称为 world）</font>。<font color='green'>而 new_group 的作用是从这个全局进程中挑选出一部分进程（ranks），组建一个独立的通信小组。</font>这在进行细粒度的并行策略（如模型并行、流水线并行）或特定的集合通信操作时至关重要。

## 🛠️ 核心参数解析
结合你之前提供的 create_group 封装代码，以下是 new_group 各个参数的详细解释：
- ranks (list[int])：必填参数。一个包含进程编号（rank）的列表，<font color='green'>用于指定哪些进程属于这个新创建的组。例如 [0, 1] 表示只包含 0 号和 1 号进程。</font>
- backend (str, optional)：<font color='red'>指定该组使用的通信后端。</font>常见的有 'nccl'（专为 NVIDIA GPU 设计，性能最佳）和 'gloo'（常用于 CPU 或作为后备选项）。<font color='green'>如果不指定，默认沿用全局进程组的后端。</font>
- timeout (timedelta, optional)：该进程组内集合通信操作的超时时间。默认通常是 30 分钟，且该参数主要适用于 gloo 后端。
- pg_options (optional)：针对特定后端的高级初始化选项（例如配置 NCCL 的分块大小等）。
- group_desc (str, optional)：<font color='green'>进程组的描述信息。这正是 PyTorch 2.4.0 引入的新特性，方便开发者在调试时识别不同的进程组。</font>

## ⚠️ 极其重要的使用规则
在使用 new_group 时，必须严格遵守以下两条铁律，否则极易导致程序死锁（Hang住）或 NCCL 报错：
- 全员参与：<font color='red'>即使某个进程不属于这个新组，它也必须执行 new_group 这行代码。</font>
- 顺序一致：<font color='blue'>所有进程中，创建各个进程组的顺序必须完全相同。</font>

### 为什么？
- <font color='red'>因为创建进程组时，底层通信库（如 NCCL）需要生成一个唯一的通信 ID，并通过共享存储在所有进程间进行同步。如果部分进程跳过了创建步骤，会导致通信 ID 不一致，从而引发死锁或运行时错误。</font>

## 💡 实战举例
1. 基础用法：创建并使用一个通信组
假设我们有 4 个进程（rank 0-3），我们只想让 rank 0 和 1 组成一个小组进行内部通信：

In [ ]:
import torch.distributed as dist

# 假设已经执行了 dist.init_process_group(...) 初始化
rank = dist.get_rank()

# 【正确示范】所有进程都必须执行这行代码！
my_group = dist.new_group(ranks=[0, 1], backend='nccl')

if rank in [0, 1]:
    # 只有 rank 0 和 1 会进入这里，进行组内通信
    tensor = torch.ones(1).cuda()
    # 这里的 all_reduce 只会在 rank 0 和 1 之间进行，不会等待 rank 2 和 3
    dist.all_reduce(tensor, group=my_group)
    print(f"Rank {rank} finished group communication with data {tensor[0]}")
else:
    # rank 2 和 3 虽然创建了组，但不能使用该组进行通信
    # 它们可以执行其他任务，或者在默认 world 组中通信
    dist.barrier() 

2. 进阶用法：创建多个通信组
在复杂的并行策略中，我们常常需要创建多个组。为了保证顺序一致，通常采用循环的方式创建：

In [ ]:
import torch.distributed as dist

rank = dist.get_rank()
group_tp = None # 假设用于张量并行的组
group_dp = None # 假设用于数据并行的组

# 保证所有进程按照完全相同的顺序创建组
for ranks_in_group in [[0, 1], [2, 3]]:
    # 所有进程同步执行 new_group
    group = dist.new_group(ranks=ranks_in_group)
    
    # 当前进程判断自己属于哪个组，并保存对应的句柄
    if rank in ranks_in_group:
        if ranks_in_group == [0, 1]:
            group_tp = group
        elif ranks_in_group == [2, 3]:
            group_dp = group

# 之后，rank 0 和 1 可以使用 group_tp 通信，rank 2 和 3 可以使用 group_dp 通信

In [ ]:
for ranks_in_group in [[0, 1], [2, 3]]:

3. 错误用法：部分进程跳过创建（会导致死锁/报错）

In [ ]:
# 【错误示范】千万不要这样写！
if rank in [0, 1]:
    # 只有 rank 0 和 1 执行了 new_group，rank 2 和 3 直接跳过了
    # 这会导致 NCCL 通信 ID 同步失败，整个分布式任务卡死或崩溃
    my_group = dist.new_group(ranks=[0, 1])
    dist.all_reduce(tensor, group=my_group)

## 总结来说，
- torch.distributed.new_group 是实现高性能、细粒度分布式并行策略的基石。只要牢记“全员参与、顺序一致”的原则，就能灵活地驾驭它来构建复杂的分布式训练架构。

# torch.distributed.init_process_group
- 创建一个包含所有进程的全局进程组（Global Process Group）

在 PyTorch 分布式训练中，这个全局进程组通常被称为“默认进程组”（default process group）。它与你之前了解的 new_group 有以下核心区别：
## 🛠️ 核心方法：init_process_group
- init_process_group 是分布式训练的入口函数。它的作用是将所有参与训练的进程（即 world_size 中定义的所有 ranks）全部拉入同一个通信组中，完成底层的通信初始化。
- 通常，你只需要在训练脚本的最开始调用一次该方法。最常用的初始化方式是通过环境变量（init_method='env://'）：

In [ ]:
import torch
import torch.distributed as dist
import os

# 1. 设置必要的环境变量（通常由 torchrun 等启动器自动注入）
os.environ['MASTER_ADDR'] = 'localhost'
os.environ['MASTER_PORT'] = '12355'
os.environ['RANK'] = '0'
os.environ['WORLD_SIZE'] = '1'

# 2. 调用 init_process_group 创建包含所有进程的全局进程组
dist.init_process_group(backend='nccl', init_method='env://')

# 此时，全局进程组（default_pg）已经创建完毕，包含了所有 ranks

## 🆚 init_process_group 与 new_group 的对比
为了帮你更好地区分这两个概念，可以参考下表：

| 特性 | `init_process_group` | `new_group` |
| :--- | :--- | :--- |
| 核心作用 | 初始化全局通信环境，创建默认进程组 | 创建自定义子进程组，用于细粒度通信 |
| 包含的进程 | 包含所有参与训练的进程（即整个 `world`） | 仅包含 `ranks` 参数中指定的部分进程 |
| 调用时机 | 分布式脚本的最开始，且只能调用一次 | 在初始化之后，可以根据需要多次调用 |
| 底层逻辑 | 建立通信后端（如 NCCL/Gloo）的基础握手 | 在全局通信基础上，建立局部的通信域 |

## 总结：
如果你想要建立一个包含所有进程的“大群”，必须使用 init_process_group；而如果你需要在大群里拉几个“小群”进行特定的局部通信（比如模型并行中的特定层通信），才会用到你之前问的 new_group